# PRAD feature-selection notebook

This notebook preserves the original feature notebook flow and functions while adding only the changes needed for the PRAD dataset:

- supports `.tsv` and `.csv` input files
- supports standard sample-row / feature-column omics tables
- keeps compatibility with the BRCA-style transposed format
- preserves full TCGA entry IDs by default so each labeled row is treated as an independent sample
- uses the labels file as the authoritative sample list and order

Ensure you have the following packages installed: scikit-learn, openpyxl, torch-geometric


In [1]:
import numpy as np
import pandas as pd

from dataclasses import dataclass
from typing import List

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif, f_classif


# ===============================
# SETTINGS
# ===============================

modality_paths = {
    "mirna": "../mirna_data.tsv",
    "mrna": "../genes_data.tsv",
    "dna": "../methyl_data.tsv"
}

label_file = "../labels_data.tsv"

FEATURE_SELECTION_K = 512
FEATURE_SELECTION_METHOD = "mutual_info"
#FEATURE_SELECTION_METHOD = "anova"

TRAIN_FRAC = 0.8
TEST_FRAC = 0.2

IGNORE_LABEL = -1
SEED = 42


# ===============================
# UTILITIES
# ===============================


TCGA_ID_LEVEL = "entry"   # "entry" keeps the full normalized barcode, "patient" keeps first 12 chars, "sample" keeps first 15 chars


def normalize_sample_id(pid):
    """
    Normalize identifiers while preserving independent entries by default.

    For this PRAD notebook we now treat EACH label row as its own sample.
    That means:
      - TCGA-EJ-A46E-01A stays distinct from TCGA-EJ-A46E-01B
      - tumor and normal rows from the same patient both remain
      - no truncation is applied unless you explicitly change TCGA_ID_LEVEL

    Levels
    ------
    entry   : keep the full normalized identifier
    patient : keep the first 12 characters (TCGA-XX-XXXX)
    sample  : keep the first 15 characters (TCGA-XX-XXXX-YY)

    Non-TCGA identifiers are returned unchanged except periods are converted
    to dashes and surrounding whitespace is removed.
    """
    pid = str(pid).replace(".", "-").strip()
    if pid.startswith("TCGA-"):
        if TCGA_ID_LEVEL == "patient":
            pid = pid[:12]
        elif TCGA_ID_LEVEL == "sample":
            pid = pid[:15]
        elif TCGA_ID_LEVEL == "entry":
            pid = pid
        else:
            raise ValueError(f"Unsupported TCGA_ID_LEVEL: {TCGA_ID_LEVEL}")
    return pid


def looks_like_tcga_id(value):
    """
    Determine whether a value resembles a TCGA sample identifier.

    Why this function exists
    ------------------------
    Auto-detect dataset orientation. 
    TCGA datasets are not always formatted consistently. In some files,
    patient/sample IDs appear as column headers, while in others they
    appear as values in the first column.

    The feature selection pipeline needs to detect which orientation
    the dataset uses so that it can correctly interpret the data as:

        rows = patients
        columns = features

    When a dataset contains a column with TCGA identifiers, we can
    confidently treat that column as the patient/sample column.

    However, some datasets do not label this column clearly (for example,
    it may simply be the first column with values like "TCGA-XX-XXXX").
    In those cases we inspect the values themselves to determine whether
    they look like TCGA identifiers.

    TCGA sample IDs typically start with:
        TCGA-XXXX-XXXX

    Occasionally they appear with periods instead of dashes:
        TCGA.XXXX.XXXX

    This function allows the loader to detect such identifiers and infer
    that the column contains patient IDs.

    Parameters
    ----------
    value : any
        A value from a dataframe cell.

    Returns
    -------
    bool
        True if the value appears to be a TCGA identifier.
    """
    value = str(value).strip()
    return value.startswith("TCGA-") or value.startswith("TCGA.")


def detect_sample_column(df):
    """
    Attempt to identify which column contains sample/patient IDs.

    This function is necessary because different TCGA datasets store
    sample identifiers in different ways. Sometimes there is an
    explicit column called 'sample_id', while other times the first
    column simply contains TCGA IDs.

    Detection proceeds in two stages:

    1. Preferred column name matching
       If a column name clearly indicates patient/sample identity,
       return that column immediately.

    2. Content inspection
       If no obvious column name is found, inspect the first column.
       If most of the first few entries look like TCGA IDs, assume
       the first column contains patient identifiers.

    If neither rule matches, the dataset is assumed to use the
    BRCA-style format where patients are stored as column headers
    rather than rows.

    Parameters
    ----------
    df : pandas.DataFrame

    Returns
    -------
    column name or None
        Name of the detected sample column, or None if no such column
        exists (indicating patients are likely stored as columns).
    """
    # Common column names used for patient/sample identifiers
    preferred = {
        "sample id", "sample_id", "patient", "patient id", "patient_id"
    }
    # ------------------------------------------------------
    # Step 1: Check if any column name matches known sample identifiers
    # ------------------------------------------------------
    for col in df.columns:
        if str(col).strip().lower() in preferred:
            return col
    # ------------------------------------------------------
    # Step 2: Inspect the first column values
    # ------------------------------------------------------
    # Some TCGA files store patient IDs in the first column but
    # without a descriptive column name.
    first_col = df.columns[0]

    # Look at the first few entries of the first column
    first_values = df[first_col].astype(str).head(10).tolist()
    
    # Count how many resemble TCGA identifiers
    if sum(looks_like_tcga_id(v) for v in first_values) >= max(1, len(first_values) // 2):
        return first_col

    # ------------------------------------------------------
    # If no sample column detected, assume patients are columns
    # ------------------------------------------------------
    return None


def detect_label_column(df):
    """
    Identify which column contains class labels.

    Label column names vary between datasets. This function attempts
    to detect the correct column using common naming conventions.

    Detection proceeds in two stages:

    1. Preferred label column names
       Check for commonly used names such as 'label', 'subtype',
       'tissue_type' or 'pam50_subtype'.

    2. Fallback strategy
       If no known name exists, select the first column that is
       NOT the sample column.

    Parameters
    ----------
    df : pandas.DataFrame

    Returns
    -------
    column name
        Name of the detected label column.

    Raises
    ------
    ValueError
        If no valid label column can be determined.
    """

    # Common names used for labels in TCGA and similar datasets
    preferred = [
        "label", "labels", "pam50_subtype", "subtype", "tissue type", "tissue_type"
    ]
    lowered = {str(c).strip().lower(): c for c in df.columns}
    
    # ------------------------------------------------------
    # Step 1: Look for a preferred label column name
    # ------------------------------------------------------
    for name in preferred:
        if name in lowered:
            return lowered[name]

    # ------------------------------------------------------
    # Step 2: Fallback strategy
    # ------------------------------------------------------
    # If no obvious label column exists, choose the first column
    # that is not the sample column.
    for col in df.columns:
        if col != detect_sample_column(df):
            return col

    # ------------------------------------------------------
    # If no suitable label column is found, raise an error
    # ------------------------------------------------------
    raise ValueError("Could not detect label column")

@dataclass
class OmicsData:
    name: str
    patient_ids: List[str]
    X: np.ndarray


# ===============================
# FILE LOADER
# ===============================

def load_file(path):

    if path.endswith(".csv"):
        df = pd.read_csv(path)

    # Allowing input files in .tsv format
    elif path.endswith(".tsv"):
        df = pd.read_csv(path, sep="\t")

    elif path.endswith(".xlsx") or path.endswith(".xls"):
        df = pd.read_excel(path, engine="openpyxl")

    else:
        raise ValueError("Unsupported file format")

    return df


# ===============================
# LOAD OMICS DATA
# ===============================

def load_omics(path, name):

    print("Loading:", name)

    df = load_file(path)

    if str(df.columns[0]).lower().startswith("unnamed"):
        df = df.drop(columns=[df.columns[0]])

    # Attempt to detect whether the dataframe contains an explicit
    # sample/patient column. This allows the loader to handle both
    # "patients as rows" and "patients as columns" formats.
    sample_col = detect_sample_column(df)

    if sample_col is not None:
        # ----------------------------------------------------------
        # PRAD / standard format
        #
        # rows    = patients
        # columns = features
        #
        # Example:
        # sample_id | gene1 | gene2 | gene3
        # TCGA-XX   | 0.23  | 0.44  | 0.91
        #
        # In this case we extract patient IDs from the sample column
        # and use the remaining columns as the feature matrix.
        # ----------------------------------------------------------
        patient_ids = [normalize_sample_id(x) for x in df[sample_col]]
        
        # Remove the sample column so only feature columns remain
        feature_df = df.drop(columns=[sample_col])

        # Convert all values to numeric (non-numeric entries become NaN)
        X = feature_df.apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)

    else:
        # ----------------------------------------------------------
        # BRCA-style format
        #
        # rows    = features
        # columns = patients
        #
        # Example:
        # gene    TCGA-A   TCGA-B   TCGA-C
        # gene1    0.23     0.11     0.42
        # gene2    0.87     0.65     0.13
        #
        # Here the patient IDs are stored as column names. We therefore
        # treat columns as patients and transpose the matrix so the
        # resulting shape becomes:
        #
        #     rows    = patients
        #     columns = features
        # ----------------------------------------------------------
        patient_ids = [normalize_sample_id(x) for x in df.columns]
        X = df.apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32).T

    X = np.nan_to_num(X, nan=0.0, posinf=1e6, neginf=-1e6)

    print("Shape:", X.shape)

    return OmicsData(name, patient_ids, X)


modalities = []

for name, path in modality_paths.items():
    modalities.append(load_omics(path, name))


# ===============================
# LOAD LABELS FIRST
# ===============================

print("Loading labels")

label_df = load_file(label_file)

# Detect patient and label columns automatically.
patient_col = detect_sample_column(label_df)
label_col = detect_label_column(label_df)

# Keep every label row as an independent sample entry
label_df["sample"] = label_df[patient_col].apply(normalize_sample_id)
label_df["label_name"] = label_df[label_col].astype(str).str.strip()

# Build mapping from label name to integer class
subtypes = sorted(label_df["label_name"].dropna().unique())
subtype_to_label = {s: i for i, s in enumerate(subtypes)}
label_df["label"] = label_df["label_name"].map(subtype_to_label)

# Use the labels file as the authoritative sample list and preserve row order.
# This keeps all labeled entries, including aliquot-level barcodes such as
# TCGA-XX-XXXX-01A and TCGA-XX-XXXX-01B, as separate samples.
all_samples = label_df["sample"].tolist()
labels = label_df["label"].to_numpy(dtype=np.int64)

print("Total samples:", len(all_samples))
print("Classes:", len(subtypes))
print("Class mapping:", subtype_to_label)

# Helpful lookup from sample ID to row positions.
# We store a list of positions for robustness in case a modality file ever
# contains the same entry ID more than once.
sample_to_positions = {}
for idx, sample in enumerate(all_samples):
    sample_to_positions.setdefault(sample, []).append(idx)

# ===============================
# ALIGN MODALITIES TO LABEL ORDER
# ===============================

x_list = []
mask = np.zeros((len(all_samples), len(modalities)), dtype=bool)

for m_idx, mod in enumerate(modalities):

    # Initialize aligned matrix as float32 to match input feature dtype and reduce memory usage
    aligned = np.zeros((len(all_samples), mod.X.shape[1]), dtype=np.float32)

    seen_counts = {}

    for i, pid in enumerate(mod.patient_ids):
        if pid not in sample_to_positions:
            continue

        occ = seen_counts.get(pid, 0)
        positions = sample_to_positions[pid]

        # If the modality contains more repeated occurrences of a sample ID
        # than the labels file, ignore the extras.
        if occ >= len(positions):
            continue

        row_idx = positions[occ]
        aligned[row_idx] = mod.X[i]
        mask[row_idx, m_idx] = True
        seen_counts[pid] = occ + 1

    x_list.append(aligned)

print("Mask shape:", mask.shape)

missing_counts = {
    name: int((~mask[:, i]).sum())
    for i, name in enumerate(modality_paths.keys())
}
print("Missing rows per modality relative to labels:", missing_counts)



# ===============================
# TRAIN / TEST SPLIT
# ===============================

labeled_idx = np.where(labels != IGNORE_LABEL)[0]

y = labels[labeled_idx]

# Old 70/10/20 split kept below for reference.
# We now split once here and reuse these indices in the base notebook.
# sss = StratifiedShuffleSplit(n_splits=1, train_size=TRAIN_FRAC, random_state=SEED)
# train_rel, temp_rel = next(sss.split(labeled_idx, y))
# train_idx = labeled_idx[train_rel]
# temp_idx = labeled_idx[temp_rel]
# y_temp = labels[temp_idx]
# sss2 = StratifiedShuffleSplit(n_splits=1, train_size=0.5, random_state=SEED)
# val_rel, test_rel = next(sss2.split(temp_idx, y_temp))
# val_idx = temp_idx[val_rel]
# test_idx = temp_idx[test_rel]

sss = StratifiedShuffleSplit(n_splits=1, train_size=TRAIN_FRAC, random_state=SEED)
train_rel, test_rel = next(sss.split(labeled_idx, y))

train_idx = labeled_idx[train_rel]
test_idx = labeled_idx[test_rel]

print("Train:", len(train_idx))
print("Test:", len(test_idx))


# ===============================
# NORMALIZATION
# ===============================

print("Normalizing features")

normalized_x_list = []

for X in x_list:

    scaler = StandardScaler()

    scaler.fit(X[train_idx])

    X_norm = scaler.transform(X)

    normalized_x_list.append(X_norm)

print("Normalization done")


# ===============================
# FEATURE SELECTION
# ===============================

print("Running feature selection")

selected_x_list = []

for i, X in enumerate(normalized_x_list):

    X_train = X[train_idx]
    y_train = labels[train_idx]

    if FEATURE_SELECTION_METHOD == "mutual_info":
        selector = SelectKBest(mutual_info_classif, k=min(FEATURE_SELECTION_K, X.shape[1]))
    else:
        selector = SelectKBest(f_classif, k=min(FEATURE_SELECTION_K, X.shape[1]))
    
    selector.fit(X_train, y_train)

    idx = selector.get_support(indices=True)

    selected = X[:, idx]

    selected_x_list.append(selected)

    print("Modality", i, "selected features:", selected.shape)


# ===============================
# SAVE PROCESSED DATA
# ===============================

print("Saving processed files")

# CSVs from the original notebook
for name, X in zip(modality_paths.keys(), selected_x_list):

    df = pd.DataFrame(X, index=all_samples)
    df["label"] = labels

    df.to_csv("processed_" + name + ".csv", index=True)

# Base notebook-compatible Excel files
excel_names = {
    "mirna": "1_tr_processed.xlsx",
    "mrna": "2_tr_processed.xlsx",
    "dna": "3_tr_processed.xlsx"
}

for name, X in zip(modality_paths.keys(), selected_x_list):
    df = pd.DataFrame(X)
    df["label"] = labels
    df.to_excel(excel_names[name], index=False)

# Save the split once so the base notebook does not re-split.
np.savez("train_test_split_indices.npz", train_idx=train_idx, test_idx=test_idx)

# Helpful metadata for validation/debugging
train_idx_set = set(train_idx.tolist())
test_idx_set = set(test_idx.tolist())

split_df = pd.DataFrame({
    "sample": all_samples,
    "label": labels,
    "label_name": [subtypes[i] if i != IGNORE_LABEL else "UNLABELED" for i in labels],
    "has_mirna": mask[:, 0],
    "has_mrna": mask[:, 1],
    "has_dna": mask[:, 2],
    "split": [
        "train" if i in train_idx_set else
        "test" if i in test_idx_set else
        "unlabeled"
        for i in range(len(all_samples))
    ]
})

split_df.to_csv("processed_patient_metadata.csv", index=False)
label_df.to_csv("normalized_labels_used.csv", index=False)

print("Done")


Loading: mirna
Shape: (533, 1881)
Loading: mrna
Shape: (533, 60660)
Loading: dna
Shape: (533, 482421)
Loading labels
Total samples: 533
Classes: 2
Class mapping: {'Normal': 0, 'Tumor': 1}
Mask shape: (533, 3)
Missing rows per modality relative to labels: {'mirna': 0, 'mrna': 0, 'dna': 0}
Train: 426
Test: 107
Normalizing features
Normalization done
Running feature selection
Modality 0 selected features: (533, 512)
Modality 1 selected features: (533, 512)
Modality 2 selected features: (533, 512)
Saving processed files
Done
